# Are VLAs secretly kNN?

A lookup over the demos predicts the VLAs' action chunks, and their chunks sit closer to the training data than a new human demonstration does.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import utils as u

data_path = Path("../data")
figs_path = Path("figs")

In [ ]:
u.load_data(data_path)
figs_path.mkdir(exist_ok=True)

In [ ]:
rows = u.lookup_rows()
y = np.arange(len(rows))[::-1]
human = float(np.mean([r[2] for r in rows]))
rows

## Both panels in one figure

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(u.W, 0.85 + 0.27 * len(rows)), sharey=True, layout="constrained")
for yi, (_lab, r2, _d, ratio) in zip(y, rows, strict=True):
    a.barh(yi, r2, color=u.LEARNED, height=0.62, edgecolor="white")
    a.text(r2 + 0.02, yi, f"{r2:.2f}", va="center", fontsize=u.FS_SMALL - 2.5, color=u.INK)
    b.barh(yi, ratio, color=u.LEARNED, height=0.62, edgecolor="white")
    b.text(ratio + 0.02, yi, f"{ratio:.2f}", va="center", fontsize=u.FS_SMALL - 2.5, color=u.INK)
a.axvline(human, color=u.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
a.text(human - 0.01, len(rows) - 0.45, "human demos", ha="right", va="bottom", fontsize=u.FS_SMALL - 2.5, color=u.KNN)
b.axvline(1.0, color=u.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
b.text(0.99, len(rows) - 0.45, "human demos", ha="right", va="bottom", fontsize=u.FS_SMALL - 2.5, color=u.KNN)
a.set_xlim(0, 1.12)
b.set_xlim(0, 1.35)
a.set_xlabel("R² of a lookup's prediction", fontsize=u.FS_SMALL - 1)
b.set_xlabel("distance to nearest training chunk (÷ human)", fontsize=u.FS_SMALL - 1)
for ax in (a, b):
    u.hbar_axes(ax)
    ax.tick_params(axis="x", labelsize=u.FS_SMALL - 2.5)
a.set_yticks(y, [r[0] for r in rows])
a.set_ylim(-0.6, len(rows) - 0.1)
fig.savefig(figs_path / "vla_lookup.pdf")

## Two separate figures (`figure.mplstyle`)

In [ ]:
labels = [re.sub(r"π([\d.]+)", r"$\\pi_{\1}$", r[0]).replace("†", r"$^\dagger$") for r in rows]
FIGSIZE = (3.4, 0.85 + 0.27 * len(rows))  # identical dimensions for both figures

### (a) Lookup prediction R²

In [ ]:
with plt.style.context(u.STYLE):
    fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")
    for yi, (_lab, r2, _d, _ratio) in zip(y, rows, strict=True):
        ax.barh(yi, r2, color=u.LEARNED, height=0.62, edgecolor="white")
        ax.text(r2 + 0.02, yi, f"{r2:.2f}", va="center", fontsize=u.FS_SMALL - 2.5, color=u.INK)
    ax.axvline(human, color=u.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
    ax.text(human - 0.01, len(rows) - 0.45, "Ground-truth actions", ha="right", va="bottom", fontsize=u.FS_SMALL - 2.5,
            color=u.KNN)  # fmt: skip
    ax.set_xlim(0, 1.12)
    ax.set_ylim(-0.6, len(rows) - 0.1)
    ax.set_yticks(y, labels)
    ax.set_xlabel("$R^2$ of a lookup's prediction", fontsize=u.FS_SMALL - 1)
    u.hbar_axes(ax)
    ax.tick_params(axis="x", labelsize=u.FS_SMALL - 2.5)
    fig.savefig(figs_path / "vla_lookup_r2.pdf")

### (b) Distance to nearest training chunk

In [ ]:
with plt.style.context(u.STYLE):
    fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")
    for yi, (_lab, _r2, _d, ratio) in zip(y, rows, strict=True):
        ax.barh(yi, ratio, color=u.LEARNED, height=0.62, edgecolor="white")
        ax.text(ratio + 0.02, yi, f"{ratio:.2f}", va="center", fontsize=u.FS_SMALL - 2.5, color=u.INK)
    ax.axvline(1.0, color=u.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
    ax.text(0.99, len(rows) - 0.45, "Ground-truth actions", ha="right", va="bottom", fontsize=u.FS_SMALL - 2.5,
            color=u.KNN)  # fmt: skip
    ax.set_xlim(0, 1.35)
    ax.set_ylim(-0.6, len(rows) - 0.1)
    ax.set_yticks(y, labels)
    ax.set_xlabel("Distance to nearest training chunk\nrelative to ground-truth", fontsize=u.FS_SMALL - 1)
    u.hbar_axes(ax)
    ax.tick_params(axis="x", labelsize=u.FS_SMALL - 2.5)
    fig.savefig(figs_path / "vla_lookup_distance.pdf")